# Component 2 — Vowel /a/, Combined + Tuned (current best model)

Dataset: AH (`data/vowel_task/`) + Italian PVS, raw audio through the same
feature extractor (`component2_phonation/features.py`) — 179 recordings,
126 participants (95 PD / 84 HC). The Kaggle UCI vowel dataset is loaded for
comparison but **excluded from training**: its Healthy/Parkinson's split
correlates with recording duration and loudness rather than acoustic
pathology (duration alone predicts its label at ~98% balanced accuracy), so
it isn't valid training data until re-verified against its original source.

This is `train_vowel_combined_tuned.py`: SelectKBest feature selection +
RandomizedSearchCV over RF / SVM / HistGradientBoosting, under the same
participant-grouped 5-fold CV used everywhere else in this component (no
recording from the same person appears in both the train and test fold).
Saved model: `models/component2_phonation_vowel_combined_tuned.joblib`.
This is the model the API's `extract_vowel_segment=True` path uses.

**Result: 82.7% CV accuracy / 82.8% CV balanced accuracy** — the best and
most methodologically honest vowel-task baseline in this component (near-
balanced classes, age-matched Italian half, grouped CV so no participant
leaks across folds).

In [ ]:
import sys, os, json
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import pandas as pd
import matplotlib.pyplot as plt

RESULTS_DIR = os.path.join(REPO_ROOT, "results", "component2_phonation")
METRICS_PATH = os.path.join(RESULTS_DIR, "vowel_combined_tuned_metrics.json")
MODEL_PATH = os.path.join(REPO_ROOT, "models", "component2_phonation_vowel_combined_tuned.joblib")

## Headline metrics (cached)

Loads the metrics already saved by the last training run — no retraining
needed just to show the number.

In [ ]:
with open(METRICS_PATH) as f:
    metrics = json.load(f)

print(f"CV accuracy:          {metrics['cv_accuracy']:.4f}")
print(f"CV balanced accuracy: {metrics['cv_balanced_accuracy']:.4f}")
print("\nBest hyperparameters found by RandomizedSearchCV:")
for k, v in metrics["best_params"].items():
    print(f"  {k}: {v}")
if "note" in metrics:
    print(f"\nNote: {metrics['note']}")

## Full classification report + confusion matrix

Re-runs the training script to regenerate the per-class breakdown and
confusion matrix (the cached metrics file above only stores the headline
numbers). Takes under a minute — 179 recordings, 10 features, 5-fold CV.
This refreshes `models/component2_phonation_vowel_combined_tuned.joblib`
and `results/component2_phonation/vowel_combined_tuned_metrics.json`.

In [ ]:
from component2_phonation import train_vowel_combined_tuned as tuned_module
tuned_module.main()

In [ ]:
with open(METRICS_PATH) as f:
    metrics = json.load(f)

report_df = pd.DataFrame(metrics["classification_report"]).T
display(report_df)

cm = metrics["confusion_matrix"]
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1]); ax.set_xticklabels(["HC", "PD"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["HC", "PD"])
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title("Vowel combined (tuned) — confusion matrix")
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i][j], ha="center", va="center")
plt.show()

## Compare against the other Component 2 baselines

In [ ]:
comparison = pd.DataFrame([
    {"Baseline": "ReadText (stand-in)", "Recordings": 37, "Participants": 37, "CV accuracy": 0.78},
    {"Baseline": "Vowel /a/ (AH only)", "Recordings": 80, "Participants": 80, "CV accuracy": 0.68},
    {"Baseline": "Vowel /a/ (Sakar et al.)", "Recordings": 756, "Participants": 252, "CV accuracy": 0.805},
    {"Baseline": "Vowel /a/ combined, untuned", "Recordings": 179, "Participants": 126, "CV accuracy": 0.81},
    {"Baseline": "Vowel /a/ combined, tuned (this model)", "Recordings": 179, "Participants": 126, "CV accuracy": metrics["cv_accuracy"]},
])
display(comparison)

plt.figure(figsize=(8, 4))
plt.barh(comparison["Baseline"], comparison["CV accuracy"])
plt.xlabel("CV accuracy")
plt.xlim(0, 1)
plt.gca().invert_yaxis()
plt.title("Component 2 baselines — CV accuracy")
plt.tight_layout()
plt.show()

## Predict on a single recording

Same `predict()` function the FastAPI backend calls, pointed explicitly at
the tuned model (this task isn't registered under `model_path_for()`, which
only knows `"readtext"` / `"vowel"`).

In [ ]:
from component2_phonation.predict import predict

sample_path = os.path.join(REPO_ROOT, "data", "vowel_task", "raw", "PD_AH", "AH_545616858-3A749CBC-3FEB-4D35-820E-E45C3E5B9B6A.wav")
result = predict(sample_path, model_path=MODEL_PATH)
print(result["audio_path"])
print("Prediction:", result["prediction"])
print(f"P(HC) = {result['probability_hc']:.4f}   P(PD) = {result['probability_pd']:.4f}")